# 场论基础可视化：曲线积分、曲面积分、散度、旋度、散度定理与 Stokes 定理

本 notebook 用 Python 图形和数值实验介绍向量场论中的基本概念。目标不是形式化证明，而是让下面这些对象变得可见：

- 向量场如何表示局部方向和强度。
- 曲线积分如何累积沿路径的标量或做功。
- 曲面积分如何累积曲面上的标量或通量。
- 散度如何描述源和汇。
- 旋度如何描述局部旋转。
- 散度定理和 Stokes 定理如何把区域内部的信息与边界信息联系起来。


## 0. 环境检查与导入

这一节检查 notebook 所需依赖，并设置二维 Matplotlib 图形中的中文字体。若系统没有可用中文字体，代码会给出安装建议。

In [ ]:
# 导入 importlib.util，用于在真正 import 前检查包是否安装。
import importlib.util
# 导入 os，用于设置 Matplotlib 配置目录。
import os
# 导入 sys，用于显示当前 Python 版本。
import sys

# 在部分受限环境中，Matplotlib 默认配置目录可能不可写；这里指定到 /tmp。
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

# 本 notebook 需要的依赖包；都已经出现在当前 requirements.txt 中。
required_packages = ["numpy", "matplotlib", "plotly", "ipywidgets"]
# 找出当前 Python 环境中缺失的依赖。
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]

# 打印 Python 版本，帮助确认 Jupyter kernel 环境。
print(f"Python: {sys.version.split()[0]}")
# 如果依赖缺失，给出安装命令并停止执行。
if missing_packages:
    print("缺少依赖：" + ", ".join(missing_packages))
    print("请先运行：python3 -m pip install -r requirements.txt")
    raise ModuleNotFoundError("Missing packages: " + ", ".join(missing_packages))

# NumPy 负责数组、网格、向量化计算和数值积分。
import numpy as np
# Matplotlib 负责二维静态图，例如箭头场、流线、等高线。
import matplotlib.pyplot as plt
# Matplotlib 字体管理器用于检测中文字体。
from matplotlib import font_manager
# Plotly graph_objects 负责三维交互图。
import plotly.graph_objects as go
# ipywidgets 提供 Jupyter 中的滑块交互。
from ipywidgets import FloatSlider, interact

# 常见中文字体候选；不同系统能识别到的名称不同。
chinese_font_candidates = [
    "Noto Sans CJK SC",
    "Noto Sans CJK JP",
    "Noto Sans CJK TC",
    "Source Han Sans SC",
    "Source Han Sans CN",
    "WenQuanYi Micro Hei",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
]
# 读取 Matplotlib 当前字体缓存中的字体名称。
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
# 选择第一个可用中文字体。
matplotlib_chinese_font = next((name for name in chinese_font_candidates if name in available_font_names), None)

# 设置 Matplotlib 的默认图形尺寸。
plt.rcParams["figure.figsize"] = (7.2, 5.8)
# 设置字体族。
plt.rcParams["font.family"] = "sans-serif"
# 如果找到中文字体，就优先使用它。
if matplotlib_chinese_font:
    plt.rcParams["font.sans-serif"] = [matplotlib_chinese_font, "DejaVu Sans"]
# 如果没有找到，仍写入候选列表，用户安装字体后重启 kernel 即可生效。
else:
    plt.rcParams["font.sans-serif"] = chinese_font_candidates + ["DejaVu Sans"]
# 避免坐标轴负号显示为方框。
plt.rcParams["axes.unicode_minus"] = False

print("依赖检查通过。")
if matplotlib_chinese_font:
    print("Matplotlib 中文字体：" + matplotlib_chinese_font)
else:
    print("未检测到 Matplotlib 可用中文字体，二维图中的中文可能显示为方框。")
    print("解决方案 Linux/WSL: sudo apt update && sudo apt install fonts-noto-cjk")
    print("安装后重启 Jupyter kernel；如仍无效，可删除字体缓存：rm -rf ~/.cache/matplotlib")


## 0.1 绘图环境与命令速查

本 notebook 主要使用以下绘图命令。

- `ax.quiver(X, Y, U, V)`：在二维平面上画向量箭头。
- `ax.streamplot(x, y, U, V)`：画二维向量场的流线。
- `ax.contourf(X, Y, Z)`：用颜色显示散度或旋度这样的标量场。
- `go.Surface(x=X, y=Y, z=Z)`：画参数曲面。
- `go.Scatter3d(...)`：画三维曲线、边界曲线和三维向量线段。
- `interact(func, a=FloatSlider(...))`：把参数变成滑块，改变参数后自动重画。

二维图用 Matplotlib，适合讲解局部方向、流线和标量背景。三维图用 Plotly，适合旋转观察曲面、通量和边界方向。

## 1. 通用绘图与数值工具

下面的工具函数把常用任务封装起来：生成网格、数值积分、画二维向量场、计算曲线积分、计算曲面积分、在三维图中添加曲线和向量。

In [ ]:
# 生成二维网格；用于二维向量场、散度和旋度背景。
def make_2d_grid(xlim=(-2, 2), ylim=(-2, 2), n=41):
    """返回 x, y, X, Y，其中 X,Y 是二维网格。"""
    # 在 x 方向生成 n 个等距采样点。
    x = np.linspace(xlim[0], xlim[1], n)
    # 在 y 方向生成 n 个等距采样点。
    y = np.linspace(ylim[0], ylim[1], n)
    # 生成二维坐标矩阵；默认形状是 (len(y), len(x))。
    X, Y = np.meshgrid(x, y)
    # 同时返回一维坐标轴和二维网格，方便 contour 与 streamplot 使用。
    return x, y, X, Y


# 兼容不同 NumPy 版本的梯形积分函数。
def trapz(values, coords, axis=-1):
    """沿指定坐标轴做梯形积分；优先使用 np.trapezoid。"""
    # NumPy 2.x 推荐 np.trapezoid；较旧版本可以回退到 np.trapz。
    if hasattr(np, "trapezoid"):
        integrate = np.trapezoid
    else:
        integrate = np.trapz
    # 返回沿 axis 的数值积分结果。
    return integrate(values, coords, axis=axis)


# 归一化向量，避免重复写除以长度的逻辑。
def normalize_vector(v, eps=1e-12):
    """返回单位向量；如果长度接近 0，返回原向量。"""
    # 转成浮点数组，便于后续线性代数计算。
    v = np.asarray(v, dtype=float)
    # 计算欧氏长度。
    norm = np.linalg.norm(v)
    # 零向量没有稳定方向，直接返回原向量。
    if norm < eps:
        return v
    # 非零向量除以长度得到单位向量。
    return v / norm


# 数值计算二维向量场 F=(P,Q) 的散度。
def divergence_2d(P, Q, X, Y, h=1e-5):
    """中心差分近似 div F = P_x + Q_y。"""
    # 计算 P 对 x 的偏导。
    dP_dx = (P(X + h, Y) - P(X - h, Y)) / (2 * h)
    # 计算 Q 对 y 的偏导。
    dQ_dy = (Q(X, Y + h) - Q(X, Y - h)) / (2 * h)
    # 返回散度。
    return dP_dx + dQ_dy


# 数值计算二维向量场 F=(P,Q) 的旋度 z 分量。
def curl_z_2d(P, Q, X, Y, h=1e-5):
    """中心差分近似 curl_z F = Q_x - P_y。"""
    # 计算 Q 对 x 的偏导。
    dQ_dx = (Q(X + h, Y) - Q(X - h, Y)) / (2 * h)
    # 计算 P 对 y 的偏导。
    dP_dy = (P(X, Y + h) - P(X, Y - h)) / (2 * h)
    # 返回二维旋度的 z 分量。
    return dQ_dx - dP_dy


# 绘制二维向量场，可选叠加一个标量背景。
def plot_vector_field_2d(P, Q, xlim=(-2, 2), ylim=(-2, 2), n=31, density=1.0, title="", background=None, background_label=""):
    """绘制二维向量场 F=(P,Q)，并可用 contourf 显示背景标量场。"""
    # 生成二维网格。
    x, y, X, Y = make_2d_grid(xlim, ylim, n)
    # 在网格上计算向量场的两个分量。
    U = np.asarray(P(X, Y), dtype=float)
    V = np.asarray(Q(X, Y), dtype=float)
    # 计算向量长度，用于箭头颜色。
    speed = np.hypot(U, V)
    # 创建二维坐标轴。
    fig, ax = plt.subplots(figsize=(7.4, 6.2))
    # 如果提供背景标量场，就先画填色等高线。
    if background is not None:
        Z = np.asarray(background(X, Y), dtype=float)
        filled = ax.contourf(X, Y, Z, levels=24, cmap="coolwarm", alpha=0.68)
        fig.colorbar(filled, ax=ax, shrink=0.84, label=background_label)
    # 画流线，帮助观察整体流动趋势。
    ax.streamplot(x, y, U, V, color="0.55", density=density, linewidth=0.9, arrowsize=1.0)
    # 对箭头做稀疏采样，避免画面过密。
    step = max(1, n // 17)
    # 箭头方向归一化，颜色保留原向量长度。
    Uq = np.divide(U, speed, out=np.zeros_like(U), where=speed > 1e-12)
    Vq = np.divide(V, speed, out=np.zeros_like(V), where=speed > 1e-12)
    # 画二维向量箭头。
    quiver = ax.quiver(X[::step, ::step], Y[::step, ::step], Uq[::step, ::step], Vq[::step, ::step], speed[::step, ::step], cmap="viridis", pivot="mid", scale=24)
    # 添加箭头长度色条。
    fig.colorbar(quiver, ax=ax, shrink=0.84, label="|F|")
    # 设置标题和坐标轴标签。
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    # 保持横纵坐标单位长度一致。
    ax.set_aspect("equal", adjustable="box")
    # 添加浅网格。
    ax.grid(alpha=0.2)
    # 显示图形。
    plt.show()
    # 返回网格和场值，便于后续复用。
    return X, Y, U, V


# 计算标量场沿参数曲线的曲线积分。
def scalar_line_integral(f, x, y, t):
    """近似计算 ∫_C f ds。"""
    # 根据采样点计算 dx/dt。
    dx_dt = np.gradient(x, t)
    # 根据采样点计算 dy/dt。
    dy_dt = np.gradient(y, t)
    # 弧长微元 ds/dt。
    speed = np.hypot(dx_dt, dy_dt)
    # 标量场在曲线上的取值。
    values = np.asarray(f(x, y), dtype=float)
    # 对 f(r(t))*|r'(t)| 做一维积分。
    integral = trapz(values * speed, t)
    # 返回积分值、标量值和速度，供绘图使用。
    return integral, values, speed


# 计算二维向量场沿参数曲线的做功积分。
def vector_line_integral(P, Q, x, y, t):
    """近似计算 ∫_C F·dr，其中 F=(P,Q)。"""
    # 根据采样点计算 dx/dt。
    dx_dt = np.gradient(x, t)
    # 根据采样点计算 dy/dt。
    dy_dt = np.gradient(y, t)
    # 计算向量场在曲线上的两个分量。
    Px = np.asarray(P(x, y), dtype=float)
    Qy = np.asarray(Q(x, y), dtype=float)
    # 做功密度 F(r(t))·r'(t)。
    density = Px * dx_dt + Qy * dy_dt
    # 对做功密度积分。
    integral = trapz(density, t)
    # 返回积分值、做功密度和切向量分量。
    return integral, density, dx_dt, dy_dt


# 绘制标量曲线积分的采样图。
def plot_scalar_line_integral(f, x, y, t, title=""):
    """用颜色显示曲线上的 f 值，并打印 ∫ f ds。"""
    # 计算数值曲线积分。
    integral, values, _ = scalar_line_integral(f, x, y, t)
    # 创建二维图。
    fig, ax = plt.subplots(figsize=(7.0, 6.0))
    # 先画曲线骨架。
    ax.plot(x, y, color="0.45", linewidth=1.2)
    # 再用散点颜色表示 f 在曲线上的取值。
    scatter = ax.scatter(x, y, c=values, cmap="viridis", s=12)
    # 标记起点和终点。
    ax.scatter([x[0]], [y[0]], c="green", s=55, label="起点", zorder=4)
    ax.scatter([x[-1]], [y[-1]], c="black", s=55, label="终点", zorder=4)
    # 添加颜色条。
    fig.colorbar(scatter, ax=ax, shrink=0.84, label="f(r(t))")
    # 设置标题和坐标轴。
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    plt.show()
    # 打印积分结果。
    print(f"∫_C f ds ≈ {integral:.6f}")
    # 返回积分值，便于后续比较。
    return integral


# 绘制向量场做功曲线积分的采样图。
def plot_vector_line_integral(P, Q, x, y, t, title=""):
    """用背景向量场和曲线颜色显示 ∫ F·dr。"""
    # 计算曲线的包围盒。
    margin = 0.35 * max(np.ptp(x), np.ptp(y), 1.0)
    # 根据包围盒设置观察范围。
    xlim = (float(np.min(x) - margin), float(np.max(x) + margin))
    ylim = (float(np.min(y) - margin), float(np.max(y) + margin))
    # 在背景网格上计算向量场。
    grid_x, grid_y, X, Y = make_2d_grid(xlim, ylim, 29)
    U = np.asarray(P(X, Y), dtype=float)
    V = np.asarray(Q(X, Y), dtype=float)
    speed = np.hypot(U, V)
    # 计算曲线上的做功积分。
    integral, density, dx_dt, dy_dt = vector_line_integral(P, Q, x, y, t)
    # 创建二维图。
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    # 画流线。
    ax.streamplot(grid_x, grid_y, U, V, color="0.65", density=1.0, linewidth=0.9)
    # 画归一化箭头，颜色表示向量长度。
    Uq = np.divide(U, speed, out=np.zeros_like(U), where=speed > 1e-12)
    Vq = np.divide(V, speed, out=np.zeros_like(V), where=speed > 1e-12)
    quiver = ax.quiver(X[::2, ::2], Y[::2, ::2], Uq[::2, ::2], Vq[::2, ::2], speed[::2, ::2], cmap="viridis", pivot="mid", scale=24)
    fig.colorbar(quiver, ax=ax, shrink=0.84, label="|F|")
    # 用曲线颜色表示局部做功密度。
    scatter = ax.scatter(x, y, c=density, cmap="coolwarm", s=14, zorder=4)
    fig.colorbar(scatter, ax=ax, shrink=0.84, label="F·r'(t)")
    # 在若干点画切向方向。
    sample_indices = np.linspace(0, len(t) - 1, 8, dtype=int)
    tangent_scale = 0.10 * max(np.ptp(x), np.ptp(y), 1.0)
    ax.quiver(x[sample_indices], y[sample_indices], dx_dt[sample_indices], dy_dt[sample_indices], angles="xy", scale_units="xy", scale=1 / tangent_scale, color="black", width=0.005, label="切向")
    # 标记起点。
    ax.scatter([x[0]], [y[0]], c="green", s=55, label="起点", zorder=5)
    # 设置标题和坐标轴。
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)
    ax.legend(loc="best")
    plt.show()
    # 打印积分结果。
    print(f"∫_C F·dr ≈ {integral:.6f}")
    # 返回积分值，便于后续比较。
    return integral


# 在 Plotly 三维图里添加一条空间曲线。
def add_3d_curve(fig, x, y, z, name="curve", color="black", width=6):
    """向 Plotly figure 添加三维曲线。"""
    # 添加 Scatter3d 曲线图层。
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines", line={"color": color, "width": width}, name=name))


# 在 Plotly 三维图里批量添加向量线段。
def add_vector_segments_3d(fig, origins, vectors, name="vectors", color="crimson", scale=1.0, width=5):
    """用线段表示三维向量；origins 和 vectors 的形状都是 (m,3)。"""
    # 转成浮点数组。
    origins = np.asarray(origins, dtype=float)
    vectors = np.asarray(vectors, dtype=float) * scale
    # 存放用 None 分隔的多段线坐标。
    xs, ys, zs = [], [], []
    # 逐个向量转换为一段线。
    for origin, vector in zip(origins, vectors):
        end = origin + vector
        xs.extend([origin[0], end[0], None])
        ys.extend([origin[1], end[1], None])
        zs.extend([origin[2], end[2], None])
    # 添加三维线段图层。
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line={"color": color, "width": width}, name=name))


# 计算参数曲面的两个切向、带方向的面积向量和面积微元。
def surface_differentials(X, Y, Z, u, v):
    """返回 r_u、r_v、r_u×r_v 和 |r_u×r_v|。"""
    # 计算 X 对 u 和 v 的偏导。
    dX_du, dX_dv = np.gradient(X, u, v, edge_order=2)
    # 计算 Y 对 u 和 v 的偏导。
    dY_du, dY_dv = np.gradient(Y, u, v, edge_order=2)
    # 计算 Z 对 u 和 v 的偏导。
    dZ_du, dZ_dv = np.gradient(Z, u, v, edge_order=2)
    # 组装 r_u。
    ru = np.stack([dX_du, dY_du, dZ_du], axis=-1)
    # 组装 r_v。
    rv = np.stack([dX_dv, dY_dv, dZ_dv], axis=-1)
    # 叉乘得到带方向的面积向量。
    normal_cross = np.cross(ru, rv)
    # 面积微元系数是叉乘向量长度。
    dS = np.linalg.norm(normal_cross, axis=-1)
    # 返回几何量。
    return ru, rv, normal_cross, dS


# 计算标量场在参数曲面上的曲面积分。
def scalar_surface_integral(f, X, Y, Z, u, v):
    """近似计算 ∬_S f dS。"""
    # 计算面积微元。
    _, _, _, dS = surface_differentials(X, Y, Z, u, v)
    # 计算标量场在曲面上的取值。
    values = np.asarray(f(X, Y, Z), dtype=float)
    # 曲面积分的被积函数是 f(r(u,v))*|r_u×r_v|。
    integrand = values * dS
    # 先沿 v 积分，再沿 u 积分。
    integral = trapz(trapz(integrand, v, axis=1), u, axis=0)
    # 返回积分值、标量值和被积函数。
    return integral, values, integrand


# 计算向量场穿过参数曲面的通量。
def flux_surface_integral(F, X, Y, Z, u, v):
    """近似计算 ∬_S F·n dS，其中方向由 r_u×r_v 决定。"""
    # 计算带方向的面积向量。
    _, _, normal_cross, _ = surface_differentials(X, Y, Z, u, v)
    # 计算向量场在曲面上的三个分量。
    Fx, Fy, Fz = F(X, Y, Z)
    # 组装向量场数组。
    F_values = np.stack([Fx, Fy, Fz], axis=-1)
    # 通量密度是 F 与带方向面积向量的点积。
    integrand = np.sum(F_values * normal_cross, axis=-1)
    # 先沿 v 积分，再沿 u 积分。
    integral = trapz(trapz(integrand, v, axis=1), u, axis=0)
    # 返回积分值、通量密度和面积向量。
    return integral, integrand, normal_cross


# 绘制带颜色的参数曲面。
def plot_surface_with_color(X, Y, Z, color_values, title="", colorbar_title="value", colorscale="Viridis", opacity=0.9):
    """用 Plotly 绘制参数曲面，并用 surfacecolor 表示一个标量。"""
    # 创建 Plotly Figure。
    fig = go.Figure()
    # 添加曲面图层。
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, surfacecolor=color_values, colorscale=colorscale, opacity=opacity, colorbar={"title": colorbar_title}, name="曲面"))
    # 设置标题、大小和边距。
    fig.update_layout(title=title, width=900, height=650, margin={"l": 0, "r": 0, "t": 55, "b": 0})
    # 设置三维坐标轴标题和比例。
    fig.update_scenes(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data")
    # 显示图形。
    fig.show()
    # 返回 Figure，便于后续叠加图层。
    return fig


## 2. 向量场：给每一点分配一个向量

二维向量场通常写成

\[
\mathbf F(x,y)=(P(x,y),Q(x,y)).
\]

下面几个典型例子分别表现源、汇、旋转和剪切。箭头给出局部方向，流线给出整体趋势。

In [ ]:
# 源场：向外流出。
def P_source(x, y):
    return x


def Q_source(x, y):
    return y


plot_vector_field_2d(
    P_source,
    Q_source,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="源场 F(x,y)=(x,y)：向外流出",
)


# 汇场：向内流入。
def P_sink(x, y):
    return -x


def Q_sink(x, y):
    return -y


plot_vector_field_2d(
    P_sink,
    Q_sink,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="汇场 F(x,y)=(-x,-y)：向内流入",
)


In [ ]:
# 旋转场：绕原点逆时针旋转。
def P_rotation(x, y):
    return -y


def Q_rotation(x, y):
    return x


plot_vector_field_2d(
    P_rotation,
    Q_rotation,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="旋转场 F(x,y)=(-y,x)：绕原点旋转",
)


# 剪切场：水平方向速度随 y 改变。
def P_shear(x, y):
    return y


def Q_shear(x, y):
    return 0 * x


plot_vector_field_2d(
    P_shear,
    Q_shear,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="剪切场 F(x,y)=(y,0)：上下两侧方向相反",
)


## 3. 曲线积分：沿一条路径累积

曲线积分有两种常见形式。

标量场沿曲线的积分：

\[
\int_C f\,ds=\int_a^b f(r(t))\lVert r'(t)\rVert\,dt.
\]

向量场沿曲线的做功积分：

\[
\int_C \mathbf F\cdot d\mathbf r=\int_a^b \mathbf F(r(t))\cdot r'(t)\,dt.
\]

第一种关心曲线经过哪里以及弧长，第二种还关心路径方向。

In [ ]:
# 生成一条单位圆曲线。
t = np.linspace(0, 2 * np.pi, 800)
# 单位圆的 x 坐标。
x_circle = np.cos(t)
# 单位圆的 y 坐标。
y_circle = np.sin(t)


# 定义一个标量场；离原点越远数值越大，并叠加轻微 x 方向变化。
def scalar_field_for_curve(x, y):
    return 1 + 0.35 * x + 0.25 * (x**2 + y**2)


plot_scalar_line_integral(
    scalar_field_for_curve,
    x_circle,
    y_circle,
    t,
    title="标量曲线积分：沿单位圆累积 f ds",
)


In [ ]:
# 对旋转场 F=(-y,x) 沿单位圆逆时针做功。
work_circle = plot_vector_line_integral(
    P_rotation,
    Q_rotation,
    x_circle,
    y_circle,
    t,
    title="向量曲线积分：旋转场沿单位圆的做功",
)

# 解析值是 2π，因为 F(r(t)) 与 r'(t) 完全同向且长度为 1。
print(f"解析参考值 2π ≈ {2 * np.pi:.6f}")
print(f"误差 ≈ {abs(work_circle - 2 * np.pi):.3e}")


In [ ]:
# 保守场例子：F=∇(x²+y²)=(2x,2y)。
def P_gradient(x, y):
    return 2 * x


def Q_gradient(x, y):
    return 2 * y


# 路径 A：从 (0,0) 到 (1,1) 的直线。
s = np.linspace(0, 1, 600)
x_straight = s
y_straight = s

# 路径 B：先从 (0,0) 到 (1,0)，再从 (1,0) 到 (1,1)。
x_corner = np.where(s <= 0.5, 2 * s, 1.0)
y_corner = np.where(s <= 0.5, 0.0, 2 * s - 1.0)

# 分别计算两个路径上的做功。
work_straight, _, _, _ = vector_line_integral(P_gradient, Q_gradient, x_straight, y_straight, s)
work_corner, _, _, _ = vector_line_integral(P_gradient, Q_gradient, x_corner, y_corner, s)

# 打印结果；理论上都等于势函数差 2。
print(f"直线路径做功 ≈ {work_straight:.6f}")
print(f"折线路径做功 ≈ {work_corner:.6f}")
print("势函数差 φ(1,1)-φ(0,0)=2")


## 4. 曲面积分：在一张曲面上累积

曲面积分也有两种常见形式。

标量场在曲面上的积分：

\[
\iint_S f\,dS=\iint_D f(r(u,v))\lVert r_u\times r_v\rVert\,du\,dv.
\]

向量场穿过曲面的通量：

\[
\iint_S \mathbf F\cdot \mathbf n\,dS=\iint_D \mathbf F(r(u,v))\cdot (r_u\times r_v)\,du\,dv.
\]

通量需要选择曲面方向；方向由参数化中的 \(r_u\times r_v\) 决定。

In [ ]:
# 构造一个抛物面片 r(u,v)=(u,v,0.3u²+0.15v²)。
u = np.linspace(-1.4, 1.4, 90)
v = np.linspace(-1.4, 1.4, 90)
U, V = np.meshgrid(u, v, indexing="ij")
X_para = U
Y_para = V
Z_para = 0.3 * U**2 + 0.15 * V**2


# 定义曲面上的标量场。
def scalar_field_surface(x, y, z):
    return 1 + 0.35 * z


# 计算标量曲面积分。
surface_integral, surface_values, surface_integrand = scalar_surface_integral(
    scalar_field_surface,
    X_para,
    Y_para,
    Z_para,
    u,
    v,
)

# 用颜色显示 f 在曲面上的取值。
plot_surface_with_color(
    X_para,
    Y_para,
    Z_para,
    surface_values,
    title="标量曲面积分：抛物面片上的 f(x,y,z)",
    colorbar_title="f",
)

print(f"∬_S f dS ≈ {surface_integral:.6f}")


In [ ]:
# 构造一个倾斜平面片 r(u,v)=(u,v,0.3u)。
u = np.linspace(-1, 1, 80)
v = np.linspace(-1, 1, 80)
U, V = np.meshgrid(u, v, indexing="ij")
X_plane = U
Y_plane = V
Z_plane = 0.3 * U


# 定义竖直向上的常向量场。
def upward_field(x, y, z):
    return 0 * x, 0 * y, 1 + 0 * z


# 计算穿过平面片的通量；方向由 r_u×r_v 决定。
flux_plane, flux_density_plane, normal_cross_plane = flux_surface_integral(
    upward_field,
    X_plane,
    Y_plane,
    Z_plane,
    u,
    v,
)

# 用颜色显示通量密度 F·(r_u×r_v)。
fig = plot_surface_with_color(
    X_plane,
    Y_plane,
    Z_plane,
    flux_density_plane,
    title="向量曲面积分：竖直场穿过倾斜平面片的通量",
    colorbar_title="F·(r_u×r_v)",
    colorscale="RdBu",
)

# 在曲面中心附近画几个方向面积向量。
sample = np.s_[10:-10:18, 10:-10:18]
origins = np.stack([X_plane[sample].ravel(), Y_plane[sample].ravel(), Z_plane[sample].ravel()], axis=1)
vectors = normal_cross_plane[sample].reshape(-1, 3)
vectors = np.array([normalize_vector(vec) for vec in vectors])
add_vector_segments_3d(fig, origins, vectors, name="曲面方向", color="black", scale=0.28)
fig.show()

print(f"∬_S F·n dS ≈ {flux_plane:.6f}")
print("解析参考值：这个例子中通量等于 xy 投影面积 4")


## 5. 散度：局部源汇强度

二维向量场 \(\mathbf F=(P,Q)\) 的散度是

\[
\nabla\cdot\mathbf F=\frac{\partial P}{\partial x}+\frac{\partial Q}{\partial y}.
\]

直观上，散度为正表示附近像源一样流出，散度为负表示附近像汇一样流入，散度接近零表示局部没有净流出。

In [ ]:
# 源场的散度为 2。
plot_vector_field_2d(
    P_source,
    Q_source,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="散度背景：F=(x,y) 的散度为正",
    background=lambda X, Y: divergence_2d(P_source, Q_source, X, Y),
    background_label="div F",
)

# 旋转场的散度为 0。
plot_vector_field_2d(
    P_rotation,
    Q_rotation,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="散度背景：F=(-y,x) 的散度为 0",
    background=lambda X, Y: divergence_2d(P_rotation, Q_rotation, X, Y),
    background_label="div F",
)


In [ ]:
# 鞍形线性场：右侧流出、上方流入，整体散度为 0。
def P_saddle_field(x, y):
    return x


def Q_saddle_field(x, y):
    return -y


plot_vector_field_2d(
    P_saddle_field,
    Q_saddle_field,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="散度背景：F=(x,-y) 局部拉伸但净散度为 0",
    background=lambda X, Y: divergence_2d(P_saddle_field, Q_saddle_field, X, Y),
    background_label="div F",
)


## 6. 旋度：局部旋转强度

三维旋度是一个向量。对于二维场 \(\mathbf F=(P,Q)\)，通常观察它的 \(z\) 分量：

\[
(\nabla\times\mathbf F)_z=\frac{\partial Q}{\partial x}-\frac{\partial P}{\partial y}.
\]

旋度为正表示局部倾向于逆时针旋转，旋度为负表示局部倾向于顺时针旋转。

In [ ]:
# 旋转场 F=(-y,x) 的二维旋度 z 分量为 2。
plot_vector_field_2d(
    P_rotation,
    Q_rotation,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="旋度背景：F=(-y,x) 的 curl_z 为正",
    background=lambda X, Y: curl_z_2d(P_rotation, Q_rotation, X, Y),
    background_label="curl_z F",
)

# 梯度场 F=(2x,2y) 的旋度为 0。
plot_vector_field_2d(
    P_gradient,
    Q_gradient,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="旋度背景：梯度场 F=(2x,2y) 的 curl_z 为 0",
    background=lambda X, Y: curl_z_2d(P_gradient, Q_gradient, X, Y),
    background_label="curl_z F",
)


## 7. 散度定理：体内源汇与边界通量

散度定理把三维区域内部的散度积分与边界曲面的通量联系起来：

\[
\iiint_E \nabla\cdot\mathbf F\,dV=\iint_{\partial E}\mathbf F\cdot\mathbf n\,dS.
\]

下面用单位球和径向场 \(\mathbf F=(x,y,z)\) 做数值实验。此时 \(\nabla\cdot\mathbf F=3\)，体积分是 \(3\cdot\frac{4\pi}{3}=4\pi\)。球面上 \(\mathbf F\) 正好等于外法向单位向量，所以边界通量也是 \(4\pi\)。

In [ ]:
# 单位球参数化：r(u,v)=(sin u cos v, sin u sin v, cos u)。
u = np.linspace(0, np.pi, 120)
v = np.linspace(0, 2 * np.pi, 160)
U, V = np.meshgrid(u, v, indexing="ij")
X_sphere = np.sin(U) * np.cos(V)
Y_sphere = np.sin(U) * np.sin(V)
Z_sphere = np.cos(U)


# 三维径向场 F=(x,y,z)。
def radial_3d_field(x, y, z):
    return x, y, z


# 用通量积分计算球面边界上的通量。
flux_sphere, flux_density_sphere, normal_cross_sphere = flux_surface_integral(
    radial_3d_field,
    X_sphere,
    Y_sphere,
    Z_sphere,
    u,
    v,
)

# 体内散度积分的解析值。
volume_integral_div = 4 * np.pi

# 绘制球面，并用颜色显示通量密度。
fig = plot_surface_with_color(
    X_sphere,
    Y_sphere,
    Z_sphere,
    flux_density_sphere,
    title="散度定理例子：径向场穿过单位球面的通量",
    colorbar_title="F·(r_u×r_v)",
    colorscale="Viridis",
    opacity=0.82,
)

# 选取若干球面点，画出向外的径向向量。
sample = np.s_[12:-12:24, 0:-1:32]
origins = np.stack([X_sphere[sample].ravel(), Y_sphere[sample].ravel(), Z_sphere[sample].ravel()], axis=1)
vectors = origins.copy()
vectors = np.array([normalize_vector(vec) for vec in vectors])
add_vector_segments_3d(fig, origins, vectors, name="F=(x,y,z)", color="crimson", scale=0.28)
fig.show()

print(f"边界通量 ∬_∂E F·n dS ≈ {flux_sphere:.6f}")
print(f"体内散度积分 ∭_E div F dV = 4π ≈ {volume_integral_div:.6f}")
print(f"误差 ≈ {abs(flux_sphere - volume_integral_div):.3e}")


## 8. Stokes 定理：曲面旋度通量与边界环流

Stokes 定理把曲面上的旋度通量与边界曲线上的环流联系起来：

\[
\iint_S (\nabla\times\mathbf F)\cdot\mathbf n\,dS=\oint_{\partial S}\mathbf F\cdot d\mathbf r.
\]

下面取单位圆盘 \(S\) 和向量场 \(\mathbf F=(-y,x,0)\)。它的旋度是 \((0,0,2)\)。圆盘面积是 \(\pi\)，所以曲面旋度通量为 \(2\pi\)。边界单位圆上的环流也应为 \(2\pi\)。

In [ ]:
# 单位圆盘用极坐标参数化：r(ρ,θ)=(ρcosθ, ρsinθ, 0)。
rho = np.linspace(0, 1, 90)
theta = np.linspace(0, 2 * np.pi, 180)
RHO, THETA = np.meshgrid(rho, theta, indexing="ij")
X_disk = RHO * np.cos(THETA)
Y_disk = RHO * np.sin(THETA)
Z_disk = 0 * X_disk


# 旋转场的三维版本 F=(-y,x,0)。
def rotation_3d_field(x, y, z):
    return -y, x, 0 * z


# 它的旋度是常向量 (0,0,2)。
def curl_rotation_3d(x, y, z):
    return 0 * x, 0 * y, 2 + 0 * z


# 计算曲面上的旋度通量。
curl_flux_disk, curl_flux_density, normal_cross_disk = flux_surface_integral(
    curl_rotation_3d,
    X_disk,
    Y_disk,
    Z_disk,
    rho,
    theta,
)

# 计算边界单位圆上的环流。
t_boundary = np.linspace(0, 2 * np.pi, 900)
x_boundary = np.cos(t_boundary)
y_boundary = np.sin(t_boundary)
z_boundary = 0 * t_boundary
circulation, _, _, _ = vector_line_integral(P_rotation, Q_rotation, x_boundary, y_boundary, t_boundary)

# 绘制圆盘、边界方向和边界上的向量场。
fig = plot_surface_with_color(
    X_disk,
    Y_disk,
    Z_disk,
    curl_flux_density,
    title="Stokes 定理例子：圆盘上的旋度通量与边界环流",
    colorbar_title="curl F · (r_ρ×r_θ)",
    colorscale="Viridis",
    opacity=0.74,
)
add_3d_curve(fig, x_boundary, y_boundary, z_boundary, name="边界 ∂S", color="black", width=7)

# 在边界上采样若干向量 F=(-y,x,0)，它们沿逆时针方向。
indices = np.linspace(0, len(t_boundary) - 1, 16, endpoint=False, dtype=int)
origins = np.stack([x_boundary[indices], y_boundary[indices], z_boundary[indices]], axis=1)
vectors = np.stack([-y_boundary[indices], x_boundary[indices], 0 * z_boundary[indices]], axis=1)
vectors = np.array([normalize_vector(vec) for vec in vectors])
add_vector_segments_3d(fig, origins, vectors, name="边界上的 F", color="crimson", scale=0.23)

# 在圆盘中心附近画法向量，表示正向曲面方向。
normal_origins = np.array([[0.0, 0.0, 0.0]])
normal_vectors = np.array([[0.0, 0.0, 1.0]])
add_vector_segments_3d(fig, normal_origins, normal_vectors, name="曲面正向", color="royalblue", scale=0.45, width=7)
fig.show()

print(f"曲面旋度通量 ∬_S curl F·n dS ≈ {curl_flux_disk:.6f}")
print(f"边界环流 ∮_∂S F·dr ≈ {circulation:.6f}")
print(f"解析参考值 2π ≈ {2 * np.pi:.6f}")
print(f"两边差值 ≈ {abs(curl_flux_disk - circulation):.3e}")


## 9. 交互实验

下面的交互控件用于观察参数变化对场形状、散度和旋度的影响。

In [ ]:
# 交互实验 1：同时调节源汇强度 a 和旋转强度 b。
def explore_source_rotation(a=0.6, b=1.0):
    # 定义组合场 F=(a x - b y, a y + b x)。
    def P_mix(x, y):
        return a * x - b * y

    def Q_mix(x, y):
        return a * y + b * x

    # 对这个线性场，散度是 2a，旋度是 2b。
    print(f"理论散度 div F = {2 * a:.2f}")
    print(f"理论旋度 curl_z F = {2 * b:.2f}")
    plot_vector_field_2d(
        P_mix,
        Q_mix,
        xlim=(-2, 2),
        ylim=(-2, 2),
        title=f"组合场：源汇强度 a={a:.2f}，旋转强度 b={b:.2f}",
        background=lambda X, Y: divergence_2d(P_mix, Q_mix, X, Y),
        background_label="div F",
    )


interact(
    explore_source_rotation,
    a=FloatSlider(value=0.6, min=-1.5, max=1.5, step=0.1, description="a"),
    b=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description="b"),
);


In [ ]:
# 交互实验 2：改变圆半径，观察旋转场 F=(-y,x) 的环流。
def explore_circulation_radius(radius=1.0):
    # 生成半径为 radius 的圆。
    t = np.linspace(0, 2 * np.pi, 900)
    x = radius * np.cos(t)
    y = radius * np.sin(t)
    # 计算环流。
    circulation = plot_vector_line_integral(
        P_rotation,
        Q_rotation,
        x,
        y,
        t,
        title=f"半径 R={radius:.2f} 的圆周环流",
    )
    # 对 F=(-y,x)，理论环流是 2πR²。
    print(f"理论值 2πR² ≈ {2 * np.pi * radius**2:.6f}")
    print(f"误差 ≈ {abs(circulation - 2 * np.pi * radius**2):.3e}")


interact(
    explore_circulation_radius,
    radius=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description="R"),
);


## 10. 小练习

下面的代码块给出几个可以直接修改的练习。

1. 改变 `P_student` 和 `Q_student`，观察散度与旋度背景。
2. 改变闭曲线半径或方向，比较环流如何变化。
3. 改变曲面参数化，观察曲面方向改变时通量符号如何改变。


In [ ]:
# 练习 1：自定义二维向量场。
def P_student(x, y):
    return x - 0.6 * y


def Q_student(x, y):
    return 0.6 * x - y


plot_vector_field_2d(
    P_student,
    Q_student,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="练习 1：自定义场的散度背景",
    background=lambda X, Y: divergence_2d(P_student, Q_student, X, Y),
    background_label="div F",
)

plot_vector_field_2d(
    P_student,
    Q_student,
    xlim=(-2, 2),
    ylim=(-2, 2),
    title="练习 1：自定义场的旋度背景",
    background=lambda X, Y: curl_z_2d(P_student, Q_student, X, Y),
    background_label="curl_z F",
)


In [ ]:
# 练习 2：把同一个圆反向走一遍，观察环流符号。
t_forward = np.linspace(0, 2 * np.pi, 800)
t_backward = np.linspace(2 * np.pi, 0, 800)

x_forward = np.cos(t_forward)
y_forward = np.sin(t_forward)
x_backward = np.cos(t_backward)
y_backward = np.sin(t_backward)

work_forward, _, _, _ = vector_line_integral(P_rotation, Q_rotation, x_forward, y_forward, t_forward)
work_backward, _, _, _ = vector_line_integral(P_rotation, Q_rotation, x_backward, y_backward, t_forward)

print(f"逆时针环流 ≈ {work_forward:.6f}")
print(f"顺时针环流 ≈ {work_backward:.6f}")
print("方向反转时，向量曲线积分符号也会反转。")
